# Boundary disagreement QC

Existing outputs are used to test whether label disagreement is concentrated near ROI boundaries or persists inside ROIs.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
RESULT_ROOT=Path('/mnt/newStor/paros/paros_WORK/xintong/results/native_surface_analysis')
TEMPLATE='fsaverage6'
sf=sorted(RESULT_ROOT.glob(f'boundary_disagreement_{TEMPLATE}_D*/tables/boundary_agreement_summary.csv'))
hf=sorted(RESULT_ROOT.glob(f'boundary_disagreement_{TEMPLATE}_D*/tables/boundary_agreement_by_hop.csv'))
summary=pd.concat([pd.read_csv(p) for p in sf],ignore_index=True)
by_hop=pd.concat([pd.read_csv(p) for p in hf],ignore_index=True)
print(f'subjects={summary.subject_id.nunique():,}; rows={len(summary):,}; template={TEMPLATE}')
display(summary.head())

## Boundary-near versus internal disagreement

Boundary-near means ≤2 native-surface hops.

In [ ]:
cohort=summary.agg(valid_mappings=('valid_mappings','sum'),disagreements=('disagreements','sum'),boundary_near_mappings=('boundary_near_mappings','sum'),boundary_near_disagreements=('boundary_near_disagreements','sum'),internal_mappings=('internal_mappings','sum'),internal_disagreements=('internal_disagreements','sum'))
cohort['overall_disagreement_pct']=100*cohort.disagreements/cohort.valid_mappings
cohort['boundary_near_disagreement_pct']=100*cohort.boundary_near_disagreements/cohort.boundary_near_mappings
cohort['internal_disagreement_pct']=100*cohort.internal_disagreements/cohort.internal_mappings
display(cohort.to_frame().T.round(3))

In [ ]:
plot_df=pd.DataFrame({'Location':['Boundary-near (≤2 hops)','Internal (>2 hops)'],'Disagreement (%)':[cohort.boundary_near_disagreement_pct,cohort.internal_disagreement_pct]})
fig,ax=plt.subplots(figsize=(4,3)); sns.barplot(data=plot_df,x='Location',y='Disagreement (%)',color='#0072B2',ax=ax); ax.set_title('Label disagreement by surface location'); ax.set_xlabel(''); ax.spines[['top','right']].set_visible(False); fig.tight_layout(); plt.show()

## Low-agreement subject × region screening

In [ ]:
low_agreement=summary.sort_values('disagreement_pct',ascending=False).head(30)
display(low_agreement[['subject_id','hemisphere','region','valid_mappings','disagreement_pct','boundary_near_disagreement_pct','internal_disagreement_pct','disagreements_near_boundary_pct']].round(2))
fig,ax=plt.subplots(figsize=(5,4)); sns.scatterplot(data=summary,x='internal_disagreement_pct',y='boundary_near_disagreement_pct',hue='hemisphere',alpha=.55,s=22,ax=ax); ax.set_xlabel('Internal disagreement (%)'); ax.set_ylabel('Boundary-near disagreement (%)'); ax.set_title('Boundary versus internal disagreement'); ax.spines[['top','right']].set_visible(False); plt.show()

## Surface-hop profile

In [ ]:
hop=by_hop.groupby('hop_bin',as_index=False).agg(valid_mappings=('valid_mappings','sum'),disagreements=('disagreements','sum'))
hop['disagreement_pct']=100*hop.disagreements/hop.valid_mappings
display(hop.round(2))
fig,ax=plt.subplots(figsize=(4,3)); sns.lineplot(data=hop,x='hop_bin',y='disagreement_pct',marker='o',color='#D55E00',ax=ax); ax.set_xlabel('Distance from native ROI boundary (surface hops)'); ax.set_ylabel('Disagreement (%)'); ax.spines[['top','right']].set_visible(False); plt.show()

## Medial-wall and unknown-label checks

In [ ]:
display(summary[['invalid_target_labels','unknown_native_labels']].sum().to_frame('count').T)
print('Rows with invalid target labels:',int((summary.invalid_target_labels>0).sum()))
print('Rows with unknown native labels:',int((summary.unknown_native_labels>0).sum()))

## Interpretation

Boundary-near disagreement substantially above internal disagreement supports a boundary-driven explanation. Elevated internal disagreement indicates a possible mapping, annotation, or label-assignment problem and should be visually inspected.